#✅ Practical: Named Entity Recognition (NER)

#👨‍🏫 Objective:
Train a simple NER model using spaCy on a small custom dataset (rule-based / statistical training, no transformer).

🔧 Step 1: Install spaCy

In [8]:
!pip install spacy


#📚 Step 2: Prepare the Training Data
NER training data format in spaCy:

In [9]:
TRAIN_DATA = [
    ("Google was founded by Larry Page.", {"entities": [(0, 6, "ORG"), (24, 34, "PERSON")]}),
    ("Barack Obama was born in Hawaii.", {"entities": [(0, 12, "PERSON"), (25, 31, "GPE")]}),
    ("Amazon is based in Seattle.", {"entities": [(0, 6, "ORG"), (20, 27, "GPE")]}),
    ("Elon Musk is the CEO of SpaceX.", {"entities": [(0, 9, "PERSON"), (27, 33, "ORG")]}),
]


#🧠 Step 3: Create and Train spaCy NER Model

In [10]:
# Import required modules
import spacy
from spacy.training.example import Example

# Step 1: Create a blank English NLP model
nlp = spacy.blank("en")  # No pre-trained components used

# Step 2: Add the Named Entity Recognizer to the pipeline
ner = nlp.add_pipe("ner")

# Example training data in expected format
TRAIN_DATA = [
    ("Google was founded by Larry Page.", {"entities": [(0, 6, "ORG"), (24, 34, "PERSON")]}),
    ("Barack Obama was born in Hawaii.", {"entities": [(0, 12, "PERSON"), (25, 31, "GPE")]}),
    ("Amazon is based in Seattle.", {"entities": [(0, 6, "ORG"), (20, 27, "GPE")]}),
    ("Elon Musk is the CEO of SpaceX.", {"entities": [(0, 9, "PERSON"), (27, 33, "ORG")]}),
]

# Step 3: Add custom entity labels to the NER component
for _, annotations in TRAIN_DATA:
    for ent in annotations.get("entities"):
        ner.add_label(ent[2])  # Add each unique entity label (e.g., PERSON, ORG, GPE)

# Step 4: Disable other pipeline components during training (though in blank model only 'ner' exists)
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

# Step 5: Start training
import random
from spacy.util import minibatch

# Initialize model training
nlp.begin_training()

# Train for 30 iterations (epochs)
for itn in range(30):
    random.shuffle(TRAIN_DATA)  # Shuffle the training data each epoch
    losses = {}  # Dictionary to collect loss values
    batches = minibatch(TRAIN_DATA, size=2)  # Create mini-batches from the training data

    # Process each batch
    for batch in batches:
        for text, annotations in batch:
            doc = nlp.make_doc(text)  # Convert text to spaCy Doc object
            example = Example.from_dict(doc, annotations)  # Create training example
            nlp.update([example], losses=losses)  # Update model weights based on example

    # Print loss after each iteration
    print(f"Iteration {itn + 1} - Losses: {losses}")


/usr/local/lib/python3.11/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Elon Musk is the CEO of SpaceX." with entities "[(0, 9, 'PERSON'), (27, 33, 'ORG')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Google was founded by Larry Page." with entities "[(0, 6, 'ORG'), (24, 34, 'PERSON')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Amazon is based in Seattle." with entities "[(0, 6, 'ORG'

Iteration 1 - Losses: {'ner': np.float32(19.797846)}
Iteration 2 - Losses: {'ner': np.float32(16.662956)}
Iteration 3 - Losses: {'ner': np.float32(9.28077)}
Iteration 4 - Losses: {'ner': np.float32(6.1770926)}
Iteration 5 - Losses: {'ner': np.float32(12.725299)}
Iteration 6 - Losses: {'ner': np.float32(6.1148148)}
Iteration 7 - Losses: {'ner': np.float32(3.9471264)}
Iteration 8 - Losses: {'ner': np.float32(2.592036)}
Iteration 9 - Losses: {'ner': np.float32(0.98639655)}
Iteration 10 - Losses: {'ner': np.float32(0.13046327)}
Iteration 11 - Losses: {'ner': np.float32(0.047925077)}
Iteration 12 - Losses: {'ner': np.float32(0.008997051)}
Iteration 13 - Losses: {'ner': np.float32(0.00017529554)}
Iteration 14 - Losses: {'ner': np.float32(2.8884258e-06)}
Iteration 15 - Losses: {'ner': np.float32(3.8970256e-07)}
Iteration 16 - Losses: {'ner': np.float32(2.3054238e-07)}
Iteration 17 - Losses: {'ner': np.float32(2.1716515e-07)}
Iteration 18 - Losses: {'ner': np.float32(2.455156e-06)}
Iteration 1

#🔍 Step 4: Test the Trained Model

In [11]:
# Test with a new sentence
test_text = "Jeff Bezos founded Amazon in Washington."
doc = nlp(test_text)

print("Entities:", [(ent.text, ent.label_) for ent in doc.ents])


Entities: [('Jeff Bezos', 'PERSON'), ('Washington', 'GPE')]


#📊 Step 5: Save and Load the Model
Save the model


In [12]:
nlp.to_disk("ner_model")


Load the model

In [13]:
loaded_nlp = spacy.load("ner_model")
doc = loaded_nlp("Sundar Pichai works at Google.")
print("Entities:", [(ent.text, ent.label_) for ent in doc.ents])


Entities: [('Google', 'ORG')]


✅ Optional Step 6: Create Streamlit UI

pip install streamlit


In [14]:
%%writefile ner_streamlit.py


import streamlit as st
import spacy

st.title("Custom NER Demo")
nlp = spacy.load("ner_model")

user_input = st.text_area("Enter text for NER:")

if st.button("Analyze"):
    doc = nlp(user_input)
    for ent in doc.ents:
        st.write(f"**{ent.text}** — {ent.label_}")


Overwriting ner_streamlit.py


#📝 Summary

| Step | Description                           |
| ---- | ------------------------------------- |
| 1    | Install `spaCy`                       |
| 2    | Prepare small custom training data    |
| 3    | Create a blank spaCy model and train  |
| 4    | Evaluate with new sentences           |
| 5    | Save and load the model               |
| 6    | Optional Streamlit UI for interaction |
